# Caso Práctico: Exploración de Series Temporales

En este caso práctico veremos como realizar el análisis de series temporales empleando Python. 

Los datos corresponden con el número de pasajeros mensuales registrados en vuelos internacionales. 

El objetivo de este notebook es que es alumno sea capaz de generar los gráficos. Sin embargo, no se proporcionan las interpretaciones de los mismos.

El notebook contiene los siguientes puntos:

1. Lectura de datos.

2. Visualización de los datos.

3. Comprobación de valores nulos.

4. Aplicación de suavizado a la serie temporal y visualización de la serie resultante.

5. Visualización de la distribución de los datos.

6. Generación de gráficos estacionales.

7. Aplicación de la técnica de descomposición de Series Temporales.

## Lectura de datos

In [1]:
import datetime
import pandas as pd
import plotly.express as px

data_path = "AirPassengers.csv"

df = pd.read_csv(data_path)

df['Fecha'] = pd.to_datetime(df['Month'], format='%Y-%m')

df.set_index('Fecha', inplace=True, drop=True)
df.head()

,Month,#Passengers
Fecha,,
1949-01-01,1949-01,112
1949-02-01,1949-02,118
1949-03-01,1949-03,132
1949-04-01,1949-04,129
1949-05-01,1949-05,121


## Visualización de la serie temporal

In [2]:
fig = px.line(df,
              x=df.index,
              y='#Passengers',
              markers=False,
              labels="",
              title='Número de pasajeros por mes')

fig['layout']['xaxis1']['title'] = 'Fecha'
fig['layout']['yaxis1']['title'] = 'Número de pasajeros'

fig.show()

## Comprobación de valores nulos

In [3]:
missing_values_mask = df.isnull()

fig = px.imshow(missing_values_mask.T, 
                labels=dict(x="Time Index", y="Variables", color="Missing Values"), 
                x=df.index, 
                y=df.columns,
                color_continuous_scale=[[0, 'green'], [1, 'white']])

fig.update_layout(title="Valores nulos",
                  xaxis_title="",
                  yaxis_title="Variables",
                  xaxis_showgrid=True, yaxis_showgrid=True)

fig.show()

## Aplicación de suavizado a la serie temporal y visualización de la serie resultante

In [4]:
missing_values_mask = df.isnull()

fig = px.imshow(missing_values_mask.T, 
                labels=dict(x="Time Index", y="Variables", color="Missing Values"), 
                x=df.index, 
                y=list(df.columns),
                color_continuous_scale=[[0, 'green'], [1, 'white']])

fig.update_layout(title="Valores nulos",
                  xaxis_title="",
                  yaxis_title="Variables",
                  xaxis_showgrid=True, yaxis_showgrid=True)

fig.show()


## Visualización de la distribución de los datos.


In [5]:
fig = px.histogram(df['#Passengers'])
fig.show()

In [6]:
df["rolling_monthly_avg"] = df["#Passengers"].rolling(window=5).mean()

In [7]:
fig = px.histogram(df['rolling_monthly_avg'], title="Distribución de la media móvil mensual")
fig.update_layout(xaxis_title="Promedio de pasajeros", yaxis_title="Frecuencia")
fig.show()

In [8]:
df['Sales Differentiation'] = df["Sales"].diff(periods=1)
fig = px.histogram(df['Sales Differentiation'])
fig.show()

KeyError: 'Sales'

## Generación de gráficos estacionales.

In [ ]:
df['Month_Name'] = df.index.strftime('%B')
df['Year'] = df.index.year

In [ ]:
plot_df = df.groupby(['Year', 'Month_Name'])['#Passengers'].mean().reset_index().dropna()
fig = px.box(plot_df, y="#Passengers", x="Month_Name", log_y=True, title="Box Plot: Promedios mensuales de pasajero")
fig.update_layout(xaxis_title="Mes", yaxis_title="Pasajeros")
fig.show()

In [ ]:
interested_years = [1951, 1952, 1953]
plot_df = pd.pivot_table(df[df.Year.isin(interested_years)], index="Year", values='#Passengers', columns="Month_Name", aggfunc="mean")
plot_df.index = "Y" + plot_df.index.astype(str)

fig = px.imshow(plot_df, height=600, title="Pasajeros: Mes vs Año")
fig['layout']['xaxis1']['title']='Mes'
fig['layout']['yaxis1']['title']='Año'
fig.show()

## Aplicación de la técnica de descomposición de Series Temporales.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def decomposition_plot(
        ts_index, observed=None, seasonal=None, trend=None, resid=None
    ):
        """Plots the decomposition output
        """
        series = []
        if observed is not None:
            series += ["Original"]
        if trend is not None:
            series += ["Trend"]
        if seasonal is not None:
            series += ["Seasonal"]
        if resid is not None:
            series += ["Residual"]
        if len(series) == 0:
            raise ValueError(
                "All component flags were off. Need atleast one of the flags turned on to plot."
            )
        fig = make_subplots(
            rows=len(series), cols=1, shared_xaxes=True, subplot_titles=series
        )
        x = ts_index
        row = 1
        if observed is not None:
            fig.append_trace(
                go.Scatter(x=x, y=observed, name="Original"), row=row, col=1
            )
            row += 1
        if trend is not None:
            fig.append_trace(
                go.Scatter(x=x, y=trend, name="Trend"), row=row, col=1
            )
            row += 1
        if seasonal is not None:
            fig.append_trace(
                go.Scatter(x=x, y=seasonal, name="Seasonal"),
                row=row,
                col=1,
            )
            row += 1
        if resid is not None:
            fig.append_trace(
                go.Scatter(x=x, y=resid, name="Residual"), row=row, col=1
            )
            row += 1

        fig.update_layout(
            title_text="Seasonal Decomposition",
            autosize=False,
            width=1200,
            height=700,
            title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
            titlefont={"size": 20},
            legend_title=None,
            showlegend=False,
            legend=dict(
                font=dict(size=15),
                orientation="h",
                yanchor="bottom",
                y=0.98,
                xanchor="right",
                x=1,
            ),
            yaxis=dict(
                titlefont=dict(size=15),
                tickfont=dict(size=15),
            ),
            xaxis=dict(
                titlefont=dict(size=15),
                tickfont=dict(size=15),
            )
        )
        return fig

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

res = seasonal_decompose(df["#Passengers"], period=12, model="additive", extrapolate_trend="freq")

In [ ]:
fig = decomposition_plot(df.index, res.observed, res.seasonal, res.trend, res.resid)
fig.show()